In [ ]:
import os
import stim
import tqecd
import sinter
from typing import List
import matplotlib.pyplot as plt

In [ ]:
def noisify_phenomenological(circuit, noise = 0.001):
    noisy_circuit = stim.Circuit()

    for instruction in circuit.flattened():
        if instruction.name in ["M", "MX"]:
            noisy_circuit.append(instruction.name, instruction.targets_copy(), noise)
        elif instruction.name in ["R", "RX", "CX", "CZ", "QUBIT_COORDS", "TICK", "DETECTOR", "OBSERVABLE_INCLUDE"]:
            noisy_circuit.append(instruction)
        else:
            raise NotImplementedError(f"Incomplete noisification : {instruction.name}")

    noisy_circuit.compile_detector_sampler()
    noisy_circuit.compile_sampler()

    return noisy_circuit

In [ ]:
def noisify_circuit_level(circuit, noise = 0.001):
    noisy_circuit = stim.Circuit()

    for instruction in circuit.flattened():
        if instruction.name in ["CX", "CZ"]:
            noisy_circuit.append(instruction)
            noisy_circuit.append("DEPOLARIZE2", instruction.targets_copy(), noise)
        elif instruction.name in ["M", "MX"]:
            noisy_circuit.append(instruction.name, instruction.targets_copy(), noise)
        elif instruction.name in ["R", "RX"]:
            noisy_circuit.append(instruction)
            noisy_circuit.append("DEPOLARIZE1", instruction.targets_copy(), noise)
        elif instruction.name in ["QUBIT_COORDS", "TICK", "DETECTOR", "OBSERVABLE_INCLUDE"]:
            noisy_circuit.append(instruction)
        else:
            raise NotImplementedError(f"Incomplete noisification : {instruction.name}")

    noisy_circuit.compile_detector_sampler()
    noisy_circuit.compile_sampler()

    return noisy_circuit

In [ ]:
filename = "../assets/path/to/file.stim"
basename, _ = os.path.splitext(filename)
circuit = stim.Circuit().from_file(filename)
annotated = basename + ".annotated.stim"
if not os.path.exists(annotated):
    tqecd.annotate_detectors_automatically(circuit).to_file(annotated)

In [ ]:
circuit = stim.Circuit().from_file(annotated)
noisy = noisify_phenomenological(circuit)
errors = noisy.shortest_graphlike_error(canonicalize_circuit_errors=True)

print(f"Length of shortest graph-like error : {len(errors)}")
for error in errors:
    print(error)

In [ ]:
tasks = [
    sinter.Task(
        circuit=noisify_circuit_level(circuit, noise=noise),
        json_metadata={'d': d, 'p': noise},
    )
    for d in [5]
    for noise in [0.0001, 0.001, 0.003125, 0.00625, 0.0125, 0.025, 0.05, 0.08, 0.1]
]

collected_stats: List[sinter.TaskStats] = sinter.collect(
    num_workers=4,
    tasks=tasks,
    decoders=['pymatching'],
    max_shots=1_000_000,
    max_errors=500,
)

In [ ]:
fig, ax = plt.subplots(1, 1)
sinter.plot_error_rate(
    ax=ax,
    stats=collected_stats,
    x_func=lambda stats: stats.json_metadata['p'],
    group_func=lambda stats: stats.json_metadata['d'],
)
ax.set_ylim(1e-9, 1e-0)
ax.set_xlim(9e-4, 1.2e-1)
ax.loglog()
ax.set_title(f"Analysis of {filename}")
ax.set_xlabel("Phyical Error Rate")
ax.set_ylabel("Logical Error Rate per Shot")
ax.grid(which='major')
ax.grid(which='minor')
ax.legend()
fig.set_dpi(120)